# Auditoria de diversidade angular e energia do banco VQE

Este notebook foi preparado para uma apresentação objetiva. Ele reúne apenas os diagnósticos necessários para responder:

1. **A diversidade dos vetores \(\theta\) permanece quando tratamos os parâmetros como ângulos?**
2. **Quantas direções são necessárias para representar essa diversidade?**
3. **Existem famílias globais de soluções claramente separadas?**
4. **Todos os vetores têm a mesma qualidade energética?**
5. **Quais bitstrings estão associados às diferentes faixas de energia?**
6. **Há degenerescência física, quase-degenerescência ou redundância paramétrica?**

Para o banco atual, esperam-se aproximadamente:

- 10.000 execuções e 30 parâmetros;
- variância circular média \(\approx 0{,}185\);
- sete componentes com indício de dois modos;
- cerca de 33 componentes PCA para 90% e 40 para 95%;
- melhor KMeans em \(k=2\), mas silhouette muito baixo, perto de 0,05;
- família quase ótima ligada a `1001011000` e família subótima ligada a `1001110000`.

Esses números são apenas referências de conferência. O notebook recalcula tudo.

## Como usar

Na célula de configuração, defina `DATA_PATH` com o arquivo que contém uma linha por execução.  
Formatos aceitos: CSV, Parquet, Pickle, JSON, NPZ e NPY.

O arquivo deve conter:

- os parâmetros em colunas `theta_0`, `theta_1`, ..., ou em uma coluna vetorial;
- uma energia final por execução;
- preferencialmente um bitstring dominante por execução.

As figuras são salvas automaticamente em `outputs_auditoria_vqe/figures`.

In [ ]:
# Caso seja necessário:
# %pip install numpy pandas matplotlib scipy scikit-learn pyarrow

from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.figsize": (12, 6.5),
    "figure.dpi": 120,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "font.size": 11,
})

RANDOM_STATE = 42

# 1. Configuração

O erro energético é calculado por

\[
\Delta E_r=\left|E_r-E_{\mathrm{exato}}\right|.
\]

Para este experimento:

\[
E_{\mathrm{exato}}\approx-0.8181062577496281,
\qquad
x^\star=\texttt{1001011000}.
\]

A tolerância `ENERGY_BIN_TOL` serve somente para agrupar energias próximas. Ela não prova degenerescência física.

In [ ]:
# =========================
# CONFIGURAÇÃO PRINCIPAL
# =========================

# Exemplo:
# DATA_PATH = Path("../outputs/theta_bank.parquet")
DATA_PATH = None

# Se a detecção automática falhar, preencha:
ENERGY_COLUMN = None
BITSTRING_COLUMN = None
THETA_VECTOR_COLUMN = None

N_QUBITS = 10
EXACT_ENERGY = -0.8181062577496281
EXACT_BITSTRING = "1001011000"

TOP_BITSTRINGS = 10
TOP_EQUIVALENCE_CLASSES = 15
ENERGY_BIN_TOL = 1e-3
K_RANGE = range(2, 9)
SILHOUETTE_SAMPLE_SIZE = 3000

UNIMODAL_EXAMPLE_INDEX = 1
BIMODAL_EXAMPLE_INDEX = 11

OUTPUT_DIR = Path("outputs_auditoria_vqe")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Figuras:", FIGURE_DIR.resolve())

# 2. Carregamento e padronização

O notebook cria três objetos:

- `THETA`: matriz \(N\times p\);
- `ENERGY`: energia final de cada execução;
- `BITSTRING`: bitstring dominante.

Os ângulos são envolvidos em

\[
[-\pi,\pi)
\]

por

\[
\theta_{\mathrm{wrap}}=(\theta+\pi)\bmod 2\pi-\pi.
\]

In [ ]:
SUPPORTED_SUFFIXES = {".csv", ".parquet", ".pq", ".pkl", ".pickle", ".json", ".npz", ".npy"}

def load_any(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return json.loads(path.read_text(encoding="utf-8"))
    if suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        return {key: obj[key] for key in obj.files}
    if suffix == ".npy":
        return np.load(path, allow_pickle=True)
    raise ValueError(f"Formato não suportado: {suffix}")

def candidate_score(path):
    name = path.name.lower()
    weights = {"theta": 8, "teta": 8, "vqe": 6, "dataset": 5,
               "bank": 5, "result": 3, "energy": 3, "10000": 2}
    return sum(weight for token, weight in weights.items() if token in name)

def discover_file():
    candidates = []
    for root in [Path.cwd(), Path.cwd().parent]:
        if not root.exists():
            continue
        for path in root.iterdir():
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
                candidates.append(path)
        for directory in root.iterdir():
            if directory.is_dir() and not directory.name.startswith("."):
                try:
                    for path in directory.iterdir():
                        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
                            candidates.append(path)
                except PermissionError:
                    pass
    candidates = list(dict.fromkeys(candidates))
    candidates.sort(key=lambda p: (candidate_score(p), p.stat().st_mtime), reverse=True)
    if not candidates:
        raise FileNotFoundError("Defina DATA_PATH: nenhum arquivo compatível foi encontrado.")
    print("Arquivo selecionado automaticamente:", candidates[0])
    print("Outros candidatos:", [str(p) for p in candidates[1:6]])
    return candidates[0]

def as_dataframe(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    if isinstance(obj, dict):
        arrays = {k: v for k, v in obj.items() if np.asarray(v).ndim >= 1}
        if arrays:
            lengths = [len(np.asarray(v)) for v in arrays.values()]
            if len(set(lengths)) == 1:
                return pd.DataFrame(arrays)
        return pd.DataFrame([obj])
    if isinstance(obj, np.ndarray):
        if obj.dtype.names:
            return pd.DataFrame(obj)
        if obj.ndim == 2:
            return pd.DataFrame(obj)
        if obj.ndim == 1 and len(obj) and isinstance(obj[0], dict):
            return pd.DataFrame(list(obj))
    raise TypeError("O objeto carregado não pôde ser convertido para DataFrame.")

def parse_vector(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).ravel()
    if isinstance(value, (list, tuple)):
        return np.asarray(value, dtype=float).ravel()
    if isinstance(value, str):
        text = value.strip()
        text = re.sub(r"^array\((.*)\)$", r"\1", text)
        text = re.sub(r"np\.float\d*\(([^)]+)\)", r"\1", text)
        try:
            return np.asarray(ast.literal_eval(text), dtype=float).ravel()
        except Exception:
            numbers = re.findall(r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", text)
            if numbers:
                return np.asarray([float(x) for x in numbers], dtype=float)
    raise ValueError(f"Vetor não interpretado: {type(value)}")

def to_real(value):
    if value is None:
        return np.nan
    if isinstance(value, (float, int, complex, np.number)):
        return float(np.real(value))
    text = re.sub(r"np\.float\d*\(([^)]+)\)", r"\1", str(value).strip())
    try:
        return float(complex(text.replace(" ", "")).real)
    except Exception:
        match = re.search(r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", text)
        return float(match.group()) if match else np.nan

def clean_bitstring(value, width=None):
    if value is None:
        return None
    groups = re.findall(r"[01]+", str(value))
    if not groups:
        return None
    bit = max(groups, key=len)
    return bit.zfill(width) if width else bit

def detect_column(df, manual, candidates):
    if manual is not None:
        if manual not in df.columns:
            raise KeyError(f"Coluna não encontrada: {manual}")
        return manual
    lower = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    for col in df.columns:
        if any(candidate.lower() in str(col).lower() for candidate in candidates):
            return col
    return None

def detect_theta(df):
    if THETA_VECTOR_COLUMN is not None:
        vectors = [parse_vector(v) for v in df[THETA_VECTOR_COLUMN]]
        if len({len(v) for v in vectors}) != 1:
            raise ValueError("Vetores theta com comprimentos diferentes.")
        return np.vstack(vectors)

    indexed = []
    for col in df.columns:
        normalized = str(col).lower().replace("θ", "theta")
        match = re.search(r"(?:theta|teta)[_\[\s-]*(\d+)", normalized)
        if match:
            indexed.append((int(match.group(1)), col))
    if indexed:
        indexed.sort(key=lambda item: item[0])
        cols = [col for _, col in indexed]
        return df[cols].apply(lambda s: s.map(to_real)).to_numpy(dtype=float)

    candidates = [c for c in df.columns if any(
        token in str(c).lower()
        for token in ["theta", "teta", "optimal_point", "parameters", "params"]
    )]
    for col in candidates:
        try:
            vectors = [parse_vector(v) for v in df[col]]
            if len({len(v) for v in vectors}) == 1 and len(vectors[0]) > 1:
                return np.vstack(vectors)
        except Exception:
            continue
    raise KeyError("Theta não encontrado. Informe THETA_VECTOR_COLUMN ou use theta_0, theta_1, ...")

selected_path = Path(DATA_PATH) if DATA_PATH is not None else discover_file()
raw_df = as_dataframe(load_any(selected_path))

print("Arquivo:", selected_path)
print("Dimensão bruta:", raw_df.shape)
print("Colunas iniciais:", list(raw_df.columns[:15]))

In [ ]:
theta_matrix = detect_theta(raw_df)

energy_col = detect_column(
    raw_df, ENERGY_COLUMN,
    ["energy", "vqe_energy", "final_energy", "optimal_value",
     "objective_value", "fun", "eigenvalue"]
)
if energy_col is None:
    raise KeyError("Coluna de energia não encontrada. Defina ENERGY_COLUMN.")

bit_col = detect_column(
    raw_df, BITSTRING_COLUMN,
    ["dominant_bitstring", "best_bitstring", "most_probable_bitstring",
     "solution_bitstring", "bitstring"]
)

energy_values = raw_df[energy_col].map(to_real).to_numpy(dtype=float)

if bit_col is None:
    bitstrings = np.array([None] * len(raw_df), dtype=object)
    print("[AVISO] Bitstring não encontrado. Os gráficos correspondentes serão ignorados.")
else:
    bitstrings = raw_df[bit_col].map(
        lambda value: clean_bitstring(value, N_QUBITS)
    ).to_numpy(dtype=object)

valid = np.all(np.isfinite(theta_matrix), axis=1) & np.isfinite(energy_values)

THETA = theta_matrix[valid]
ENERGY = energy_values[valid]
BITSTRING = bitstrings[valid]
THETA_WRAPPED = (THETA + np.pi) % (2 * np.pi) - np.pi

analysis_df = pd.DataFrame({
    "row_id": np.flatnonzero(valid),
    "energy": ENERGY,
    "bitstring": BITSTRING,
})
analysis_df["absolute_energy_error"] = np.abs(ENERGY - EXACT_ENERGY)
analysis_df["relative_energy_error"] = analysis_df["absolute_energy_error"] / abs(EXACT_ENERGY)

print(f"Execuções válidas: {len(analysis_df):,}")
print(f"Parâmetros: {THETA_WRAPPED.shape[1]}")
print("Energia:", energy_col)
print("Bitstring:", bit_col)

try:
    analysis_df.to_parquet(TABLE_DIR / "analysis_dataset_standardized.parquet", index=False)
except Exception:
    analysis_df.to_csv(TABLE_DIR / "analysis_dataset_standardized.csv", index=False)

## Conferência inicial

**Pergunta respondida:** o banco carregado é o esperado?

Confira o número de execuções, o número de parâmetros e as frações de erro antes de apresentar os gráficos.

In [ ]:
summary = pd.Series({
    "n_execucoes": len(analysis_df),
    "n_parametros": THETA_WRAPPED.shape[1],
    "energia_exata": EXACT_ENERGY,
    "energia_media": analysis_df["energy"].mean(),
    "erro_medio": analysis_df["absolute_energy_error"].mean(),
    "erro_mediano": analysis_df["absolute_energy_error"].median(),
    "fracao_erro_menor_1e-3": (analysis_df["absolute_energy_error"] < 1e-3).mean(),
    "fracao_erro_menor_1e-2": (analysis_df["absolute_energy_error"] < 1e-2).mean(),
    "fracao_bitstring_exato": (analysis_df["bitstring"] == EXACT_BITSTRING).mean(),
})
display(summary.to_frame("valor"))
summary.to_csv(TABLE_DIR / "summary_global.csv", header=["valor"])

# 3. Variância circular de cada componente

### Pergunta respondida

**Quais parâmetros ficam concentrados e quais variam realmente no círculo?**

\[
C_j=\frac1N\sum_r\cos\theta_j^{(r)},
\qquad
S_j=\frac1N\sum_r\sin\theta_j^{(r)},
\]

\[
R_j=\sqrt{C_j^2+S_j^2},
\qquad
V_j=1-R_j.
\]

- \(V_j\approx0\): componente concentrada;
- \(V_j\) maior: componente angularmente mais dispersa.

Como \(V_j=1-R_j\), a concentração e a variância carregam a mesma informação em sentidos opostos.

In [ ]:
C = np.mean(np.cos(THETA_WRAPPED), axis=0)
S = np.mean(np.sin(THETA_WRAPPED), axis=0)
RESULTANT_LENGTH = np.sqrt(C**2 + S**2)
CIRCULAR_VARIANCE = 1 - RESULTANT_LENGTH
CIRCULAR_MEAN = np.arctan2(S, C)

circular_stats = pd.DataFrame({
    "theta_index": np.arange(THETA_WRAPPED.shape[1]),
    "circular_mean": CIRCULAR_MEAN,
    "resultant_length_R": RESULTANT_LENGTH,
    "circular_variance": CIRCULAR_VARIANCE,
    "linear_std": np.std(THETA_WRAPPED, axis=0),
})
circular_stats.to_csv(TABLE_DIR / "circular_statistics.csv", index=False)
display(circular_stats)

fig, ax = plt.subplots()
ax.bar(circular_stats["theta_index"], circular_stats["circular_variance"])
ax.set_title("Variância circular de cada componente de θ")
ax.set_xlabel("Índice do parâmetro")
ax.set_ylabel("Variância circular  V = 1 − R")
ax.set_xticks(np.arange(THETA_WRAPPED.shape[1]))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_variancia_circular.png", dpi=300, bbox_inches="tight")
plt.show()

print("Média:", CIRCULAR_VARIANCE.mean())
print("Máxima:", CIRCULAR_VARIANCE.max())

# 4. Multimodalidade angular

### Pergunta respondida

**Cada componente termina em uma única região angular ou em mais de uma?**

O procedimento constrói um histograma circular, suaviza-o respeitando a união entre \(-\pi\) e \(+\pi\) e conta picos suficientemente proeminentes.

- 1 modo: uma região predominante;
- 2 modos: duas regiões predominantes.

Isso descreve \(p(\theta_j)\) isoladamente. Não é o número de clusters globais nem de mínimos físicos.

In [ ]:
def circular_modes(angles, bins=72, sigma=2.0, prominence_fraction=0.08, min_distance=5):
    angles = (np.asarray(angles) + np.pi) % (2*np.pi) - np.pi
    hist, edges = np.histogram(angles, bins=bins, range=(-np.pi, np.pi), density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    smooth = gaussian_filter1d(hist.astype(float), sigma=sigma, mode="wrap")
    extended = np.tile(smooth, 3)
    prominence = max(prominence_fraction * (smooth.max() - smooth.min()), 1e-12)
    peaks, _ = find_peaks(extended, prominence=prominence, distance=min_distance)
    peaks = peaks[(peaks >= bins) & (peaks < 2*bins)] - bins
    unique = []
    for peak in peaks:
        if all(min(abs(peak-q), bins-abs(peak-q)) >= min_distance for q in unique):
            unique.append(int(peak))
    return centers, hist, smooth, np.asarray(unique, dtype=int)

MODE_DETAILS = {}
rows = []
for j in range(THETA_WRAPPED.shape[1]):
    centers, hist, smooth, peaks = circular_modes(THETA_WRAPPED[:, j])
    MODE_DETAILS[j] = dict(centers=centers, hist=hist, smooth=smooth, peaks=peaks)
    rows.append({
        "theta_index": j,
        "estimated_modes": max(1, len(peaks)),
        "peak_angles": centers[peaks].tolist() if len(peaks) else [],
    })

mode_table = pd.DataFrame(rows)
display(mode_table)
mode_table.to_json(TABLE_DIR / "angular_modes.json", orient="records", indent=2)

fig, ax = plt.subplots()
ax.bar(mode_table["theta_index"], mode_table["estimated_modes"])
ax.set_title("Diagnóstico de multimodalidade angular")
ax.set_xlabel("Índice do parâmetro")
ax.set_ylabel("Número estimado de modos")
ax.set_xticks(np.arange(THETA_WRAPPED.shape[1]))
ax.set_yticks(np.arange(1, int(mode_table["estimated_modes"].max()) + 1))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_multimodalidade.png", dpi=300, bbox_inches="tight")
plt.show()

print("Multimodais:", mode_table.loc[mode_table["estimated_modes"] > 1, "theta_index"].tolist())

## Exemplos de distribuição angular

### Pergunta respondida

**Onde uma componente termina ao longo das execuções?**

As barras são a densidade observada; a linha é a versão suavizada.  
A densidade próxima de \(-\pi\) pode ser a continuação de uma região próxima de \(+\pi\).

- \(\theta_1\): exemplo esperado de região ampla, porém única;
- \(\theta_{11}\): exemplo esperado de duas regiões.

In [ ]:
def plot_angular_distribution(j, filename):
    d = MODE_DETAILS[j]
    centers, hist, smooth, peaks = d["centers"], d["hist"], d["smooth"], d["peaks"]
    width = 2*np.pi / len(centers)
    fig, ax = plt.subplots()
    ax.bar(centers, hist, width=width, alpha=0.65, label="Histograma circular")
    ax.plot(centers, smooth, linewidth=2.2, label="Densidade suavizada")
    for p in peaks:
        ax.axvline(centers[p], linestyle="--", alpha=0.8)
    n_modes = max(1, len(peaks))
    ax.set_title(f"Distribuição angular de θ{j} ({n_modes} modo(s) estimado(s))")
    ax.set_xlabel(f"θ{j} em [−π, π)")
    ax.set_ylabel("Densidade")
    ax.set_xlim(-np.pi, np.pi)
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    print("Picos:", [float(centers[p]) for p in peaks])

if UNIMODAL_EXAMPLE_INDEX < THETA_WRAPPED.shape[1]:
    plot_angular_distribution(UNIMODAL_EXAMPLE_INDEX, "03_theta_unimodal.png")

if BIMODAL_EXAMPLE_INDEX < THETA_WRAPPED.shape[1]:
    plot_angular_distribution(BIMODAL_EXAMPLE_INDEX, "04_theta_bimodal.png")

# 5. PCA: dimensão necessária

### Pergunta respondida

**Quantos padrões lineares combinados são necessários para representar a diversidade?**

Cada ângulo é representado por:

\[
\theta_j\longrightarrow(\cos\theta_j,\sin\theta_j).
\]

O PCA trabalha no vetor de 60 coordenadas e calcula

\[
C(m)=\frac{\sum_{i=1}^{m}\lambda_i}{\sum_i\lambda_i}.
\]

O número obtido não é o número de mínimos ou soluções; ele mede a compressibilidade linear da nuvem angular.

In [ ]:
ANGULAR_EMBEDDING = np.concatenate(
    [np.cos(THETA_WRAPPED), np.sin(THETA_WRAPPED)], axis=1
)

PCA_FULL = PCA()
PCA_COORDINATES_FULL = PCA_FULL.fit_transform(ANGULAR_EMBEDDING)
explained = PCA_FULL.explained_variance_ratio_
cumulative = np.cumsum(explained)
n90 = int(np.searchsorted(cumulative, 0.90) + 1)
n95 = int(np.searchsorted(cumulative, 0.95) + 1)

pca_table = pd.DataFrame({
    "component": np.arange(1, len(explained)+1),
    "explained_variance_ratio": explained,
    "cumulative_variance": cumulative,
})
pca_table.to_csv(TABLE_DIR / "pca_variance.csv", index=False)

fig, ax = plt.subplots()
ax.plot(pca_table["component"], pca_table["cumulative_variance"], marker="o", markersize=3)
ax.axhline(0.90, linestyle="--", label="90%")
ax.axhline(0.95, linestyle=":", label="95%")
ax.axvline(n90, linestyle="--", alpha=0.7)
ax.axvline(n95, linestyle=":", alpha=0.7)
ax.set_title("Dimensão necessária para representar a diversidade angular")
ax.set_xlabel("Número de componentes principais")
ax.set_ylabel("Variância explicada acumulada")
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_pca_variancia_acumulada.png", dpi=300, bbox_inches="tight")
plt.show()

print("Componentes para 90%:", n90)
print("Componentes para 95%:", n95)
print("PC1 + PC2:", cumulative[1] if len(cumulative) > 1 else cumulative[0])

# 6. Seleção do número de clusters angulares

### Pergunta respondida

**Os vetores completos formam famílias globais claramente separadas?**

Para cada \(k\), o KMeans particiona o espaço seno–cosseno. A qualidade é medida pelo silhouette:

\[
s=\frac{b-a}{\max(a,b)}.
\]

- próximo de 1: grupos bem separados;
- próximo de 0: grupos sobrepostos;
- negativo: muitos pontos mal atribuídos.

O maior valor entre os testados não é necessariamente um bom valor absoluto.

In [ ]:
silhouette_rows = []
models = {}

for k in K_RANGE:
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(ANGULAR_EMBEDDING)
    score = silhouette_score(
        ANGULAR_EMBEDDING,
        labels,
        sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(labels)),
        random_state=RANDOM_STATE,
    )
    silhouette_rows.append({"k": k, "silhouette": score})
    models[k] = (model, labels)

silhouette_table = pd.DataFrame(silhouette_rows)
BEST_K = int(silhouette_table.loc[silhouette_table["silhouette"].idxmax(), "k"])
BEST_SILHOUETTE = float(silhouette_table["silhouette"].max())
BEST_MODEL, CLUSTER_LABELS = models[BEST_K]
silhouette_table.to_csv(TABLE_DIR / "kmeans_silhouette.csv", index=False)

fig, ax = plt.subplots()
ax.plot(silhouette_table["k"], silhouette_table["silhouette"], marker="o")
ax.scatter([BEST_K], [BEST_SILHOUETTE], s=100, label=f"Melhor k = {BEST_K}")
ax.set_title("Seleção do número de clusters angulares")
ax.set_xlabel("Número de clusters k")
ax.set_ylabel("Silhouette médio")
ax.set_xticks(list(K_RANGE))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_selecao_k.png", dpi=300, bbox_inches="tight")
plt.show()

print("Melhor k:", BEST_K)
print("Silhouette:", BEST_SILHOUETTE)

## Projeção PCA colorida pelos clusters

### Pergunta respondida

**A melhor partição aparece visualmente como ilhas separadas?**

A figura é apenas uma projeção em duas dimensões. A conclusão principal vem do silhouette calculado no espaço completo.

In [ ]:
PCA_2D = PCA(n_components=2)
PCA_2D_COORDS = PCA_2D.fit_transform(ANGULAR_EMBEDDING)

fig, ax = plt.subplots()
scatter = ax.scatter(
    PCA_2D_COORDS[:, 0], PCA_2D_COORDS[:, 1],
    c=CLUSTER_LABELS, s=10, alpha=0.45
)
ax.set_title(f"PCA angular com KMeans (k={BEST_K}, silhouette={BEST_SILHOUETTE:.3f})")
ax.set_xlabel(f"PC1 ({PCA_2D.explained_variance_ratio_[0]*100:.2f}%)")
ax.set_ylabel(f"PC2 ({PCA_2D.explained_variance_ratio_[1]*100:.2f}%)")
ax.grid(alpha=0.2)
legend = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_pca_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

# 7. Distribuição do erro energético

### Pergunta respondida

**Todas as execuções possuem a mesma qualidade?**

\[
\Delta E=\left|E_{\mathrm{VQE}}-E_{\mathrm{exato}}\right|.
\]

- próximo de zero: quase ótimo;
- segundo pico: família subótima recorrente;
- cauda: execuções de baixa qualidade.

In [ ]:
energy_error = analysis_df["absolute_energy_error"].to_numpy()

fig, ax = plt.subplots()
ax.hist(energy_error, bins=70, density=True, alpha=0.75)
ax.axvline(1e-3, linestyle="--", label=r"$10^{-3}$")
ax.axvline(1e-2, linestyle=":", label=r"$10^{-2}$")
ax.set_title("Distribuição do erro energético absoluto")
ax.set_xlabel(r"$\Delta E = |E_{\mathrm{VQE}}-E_{\mathrm{exato}}|$")
ax.set_ylabel("Densidade")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_erro_energetico.png", dpi=300, bbox_inches="tight")
plt.show()

print("Erro < 1e-3:", (energy_error < 1e-3).mean())
print("Erro < 1e-2:", (energy_error < 1e-2).mean())
print("Erro médio:", energy_error.mean())
print("Erro mediano:", np.median(energy_error))

# 8. Frequência dos bitstrings

### Pergunta respondida

**Quais soluções clássicas aparecem com maior frequência?**

Frequência não é sinônimo de qualidade. Um bitstring frequente pode ser subótimo; por isso, este gráfico deve ser apresentado junto com a energia por bitstring.

In [ ]:
valid_bits_df = analysis_df.dropna(subset=["bitstring"]).copy()

if valid_bits_df.empty:
    print("[AVISO] Nenhum bitstring válido.")
else:
    bit_counts = valid_bits_df["bitstring"].value_counts().rename_axis("bitstring").reset_index(name="count")
    bit_counts["fraction"] = bit_counts["count"] / len(valid_bits_df)
    bit_counts.to_csv(TABLE_DIR / "bitstring_frequencies.csv", index=False)

    top = bit_counts.head(TOP_BITSTRINGS).iloc[::-1]
    fig, ax = plt.subplots()
    ax.barh(top["bitstring"], top["count"])
    ax.set_title(f"{TOP_BITSTRINGS} bitstrings dominantes mais frequentes")
    ax.set_xlabel("Número de execuções")
    ax.set_ylabel("Bitstring dominante")
    ax.grid(axis="x", alpha=0.25)
    if EXACT_BITSTRING in top["bitstring"].values:
        row = top[top["bitstring"] == EXACT_BITSTRING].iloc[0]
        ax.annotate("Solução exata", xy=(row["count"], row["bitstring"]),
                    xytext=(8, 0), textcoords="offset points",
                    va="center", fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "09_frequencia_bitstrings.png", dpi=300, bbox_inches="tight")
    plt.show()

    display(bit_counts.head(20))

# 9. Energia associada a cada bitstring

### Pergunta respondida

**O mesmo bitstring aparece sempre com a mesma energia? Quais bitstrings concentram as menores energias?**

O estado VQE pode ser

\[
|\psi(\theta)\rangle=\sum_x c_x(\theta)|x\rangle,
\]

e sua energia é

\[
E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle.
\]

Por isso, o mesmo bitstring dominante pode aparecer com energias diferentes: as probabilidades dos demais estados também mudam.

In [ ]:
if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    frequent = valid_bits_df["bitstring"].value_counts().head(TOP_BITSTRINGS).index.tolist()
    subset = valid_bits_df[valid_bits_df["bitstring"].isin(frequent)]
    order = (
        subset.groupby("bitstring")["energy"]
        .agg(["mean", "count"])
        .sort_values(["mean", "count"], ascending=[True, False])
        .index.tolist()
    )
    groups = [subset.loc[subset["bitstring"] == bit, "energy"].to_numpy() for bit in order]
    rng = np.random.default_rng(RANDOM_STATE)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.boxplot(groups, tick_labels=order, showfliers=False, widths=0.55)
    for position, values in enumerate(groups, start=1):
        values_plot = rng.choice(values, 1000, replace=False) if len(values) > 1000 else values
        jitter = rng.uniform(-0.18, 0.18, len(values_plot))
        ax.scatter(position + jitter, values_plot, s=8, alpha=0.25)
    ax.axhline(EXACT_ENERGY, linestyle="--", label=f"E exata = {EXACT_ENERGY:.6f}")
    ax.set_title("Distribuição das energias por bitstring dominante")
    ax.set_xlabel("Bitstring dominante")
    ax.set_ylabel("Energia")
    ax.tick_params(axis="x", rotation=55)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "10_energia_por_bitstring.png", dpi=300, bbox_inches="tight")
    plt.show()

    bit_energy_summary = valid_bits_df.groupby("bitstring")["energy"].agg(
        count="size", mean_energy="mean", median_energy="median",
        min_energy="min", max_energy="max", std_energy="std"
    ).sort_values(["mean_energy", "count"], ascending=[True, False])
    bit_energy_summary.to_csv(TABLE_DIR / "energy_by_bitstring.csv")
    display(bit_energy_summary.head(20))

# 10. Mapa bitstring × faixa de energia

### Pergunta respondida

**Quais famílias de bitstrings ocupam cada região energética?**

As linhas são bitstrings, as colunas são faixas de energia e a intensidade representa o número de execuções.  
Este é o gráfico mais direto para mostrar a família quase ótima e a família subótima recorrente.

In [ ]:
def energy_bin(values, tolerance):
    return np.rint(np.asarray(values, dtype=float) / tolerance) * tolerance

if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    heat = valid_bits_df.copy()
    heat["energy_bin"] = energy_bin(heat["energy"], ENERGY_BIN_TOL)
    frequent = heat["bitstring"].value_counts().head(TOP_BITSTRINGS).index
    heat = heat[heat["bitstring"].isin(frequent)]
    matrix = pd.crosstab(heat["bitstring"], heat["energy_bin"])
    row_order = heat.groupby("bitstring")["energy"].mean().sort_values().index
    matrix = matrix.reindex(row_order)
    matrix = matrix.loc[:, matrix.sum(axis=0) > 0]

    fig, ax = plt.subplots(figsize=(max(12, 0.34*matrix.shape[1]), 7))
    image = ax.imshow(matrix.to_numpy(), aspect="auto", interpolation="nearest")
    ax.set_title("Mapa de frequência: bitstring × faixa de energia")
    ax.set_xlabel(f"Centro da faixa de energia (tolerância = {ENERGY_BIN_TOL:g})")
    ax.set_ylabel("Bitstring dominante")
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    positions = np.arange(len(matrix.columns))
    step = max(1, int(np.ceil(len(positions)/18)))
    shown = positions[::step]
    ax.set_xticks(shown)
    ax.set_xticklabels([f"{matrix.columns[i]:.3f}" for i in shown], rotation=60, ha="right")
    cbar = fig.colorbar(image, ax=ax)
    cbar.set_label("Número de execuções")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "11_heatmap_bitstring_energia.png", dpi=300, bbox_inches="tight")
    plt.show()

    matrix.to_csv(TABLE_DIR / "bitstring_energy_matrix.csv")

# 11. Degenerescência e redundância

## Degenerescência exata

Estados distintos possuem o mesmo autovalor:

\[
H|\psi_a\rangle=E|\psi_a\rangle,
\qquad
H|\psi_b\rangle=E|\psi_b\rangle,
\qquad
|\psi_a\rangle\neq|\psi_b\rangle.
\]

Isso exige uma verificação exata do espectro.

## Quase-degenerescência operacional

Bitstrings diferentes aparecem na mesma faixa de energia:

\[
|E_a-E_b|<\varepsilon.
\]

É dependente da tolerância \(\varepsilon\) e não prova degenerescência exata.

## Redundância paramétrica

Vetores \(\theta\) diferentes produzem aproximadamente o mesmo bitstring e a mesma energia.  
Essa é a situação mais claramente sugerida pelo banco atual.

## Bitstrings distintos por faixa de energia

### Pergunta respondida

**Quantos bitstrings diferentes aparecem dentro de cada faixa energética?**

Valores acima de 1 apontam faixas que merecem uma investigação exata de quase-degenerescência.

In [ ]:
if valid_bits_df.empty:
    print("[AVISO] Gráfico ignorado.")
else:
    deg = valid_bits_df.copy()
    deg["energy_bin"] = energy_bin(deg["energy"], ENERGY_BIN_TOL)
    operational = deg.groupby("energy_bin").agg(
        distinct_bitstrings=("bitstring", "nunique"),
        executions=("bitstring", "size"),
    ).reset_index().sort_values("energy_bin")

    sizes = 20 + 180 * operational["executions"] / operational["executions"].max()
    fig, ax = plt.subplots()
    ax.scatter(operational["energy_bin"], operational["distinct_bitstrings"],
               s=sizes, alpha=0.65)
    ax.axhline(1, linestyle="--", label="Um bitstring por faixa")
    ax.set_title("Bitstrings distintos por faixa de energia")
    ax.set_xlabel("Centro da faixa de energia")
    ax.set_ylabel("Número de bitstrings distintos")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "12_quase_degenerescencia.png", dpi=300, bbox_inches="tight")
    plt.show()

    operational.to_csv(TABLE_DIR / "near_degeneracy_operational.csv", index=False)
    display(operational[operational["distinct_bitstrings"] > 1]
            .sort_values(["distinct_bitstrings", "executions"], ascending=False).head(20))

# 12. Classes operacionais e dispersão angular

### Pergunta respondida

**Quantos vetores diferentes produzem aproximadamente o mesmo bitstring e a mesma energia?**

Uma classe é

\[
(\text{bitstring dominante},\text{faixa de energia}).
\]

A dispersão interna é a distância toroidal até a média circular da classe:

\[
d_{\mathbb T}
=
\sqrt{\sum_j
\operatorname{atan2}\left[
\sin(\theta_j-\bar\theta_j),
\cos(\theta_j-\bar\theta_j)
\right]^2}.
\]

Classe grande e angularmente espalhada é evidência de redundância paramétrica.

In [ ]:
def toroidal_distance(theta_matrix, reference):
    delta = np.arctan2(np.sin(theta_matrix-reference), np.cos(theta_matrix-reference))
    return np.linalg.norm(delta, axis=1)

if valid_bits_df.empty:
    print("[AVISO] Análise ignorada.")
else:
    classes = valid_bits_df.copy()
    classes["energy_bin"] = energy_bin(classes["energy"], ENERGY_BIN_TOL)
    decimals = max(0, int(round(-math.log10(ENERGY_BIN_TOL))))
    classes["equivalence_class"] = (
        classes["bitstring"].astype(str) + "__" +
        classes["energy_bin"].map(lambda x: f"{x:.{decimals}f}")
    )

    rows = []
    for name, group in classes.groupby("equivalence_class"):
        positions = group.index.to_numpy()
        theta_group = THETA_WRAPPED[positions]
        mean_angle = np.arctan2(
            np.mean(np.sin(theta_group), axis=0),
            np.mean(np.cos(theta_group), axis=0),
        )
        distances = toroidal_distance(theta_group, mean_angle)
        rows.append({
            "equivalence_class": name,
            "bitstring": group["bitstring"].iloc[0],
            "energy_bin": group["energy_bin"].iloc[0],
            "size": len(group),
            "mean_angular_spread": distances.mean(),
            "max_angular_spread": distances.max(),
            "mean_energy": group["energy"].mean(),
        })

    equivalence_table = pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)
    equivalence_table.to_csv(TABLE_DIR / "equivalence_classes.csv", index=False)
    display(equivalence_table.head(20))

    top = equivalence_table.head(TOP_EQUIVALENCE_CLASSES).iloc[::-1]
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(top["equivalence_class"], top["size"])
    ax.set_title("Maiores classes operacionais")
    ax.set_xlabel("Número de execuções")
    ax.set_ylabel("Bitstring + faixa de energia")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "13_classes_operacionais.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots()
    ax.scatter(equivalence_table["size"], equivalence_table["mean_angular_spread"],
               s=25, alpha=0.65)
    ax.set_title("Tamanho da classe × dispersão angular interna")
    ax.set_xlabel("Tamanho da classe")
    ax.set_ylabel("Dispersão angular média")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "14_classe_vs_dispersao.png", dpi=300, bbox_inches="tight")
    plt.show()

# 13. Erro energético × distância angular

### Pergunta respondida

**Vetores mais afastados do centro angular são necessariamente piores?**

Se não houver tendência clara, a distância geométrica entre parâmetros não é suficiente para prever a qualidade energética.

In [ ]:
distance_to_mean = toroidal_distance(THETA_WRAPPED, CIRCULAR_MEAN)
analysis_df["distance_to_circular_mean"] = distance_to_mean

fig, ax = plt.subplots()
ax.scatter(distance_to_mean, analysis_df["absolute_energy_error"], s=10, alpha=0.35)
ax.set_title("Erro energético × distância à média circular")
ax.set_xlabel("Distância toroidal à média circular")
ax.set_ylabel(r"$\Delta E$")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "15_erro_vs_distancia.png", dpi=300, bbox_inches="tight")
plt.show()

print("Correlação:", np.corrcoef(distance_to_mean, analysis_df["absolute_energy_error"])[0, 1])

# 14. Resumo automático para apresentação

A leitura recomendada é:

1. A diversidade angular existe, mas é desigual entre as componentes.
2. Alguns parâmetros possuem dois modos marginais, sem que isso prove dois mínimos físicos.
3. O PCA indica diversidade distribuída em muitas direções.
4. O KMeans só deve ser interpretado como forte se o silhouette for claramente superior a zero.
5. A energia separa soluções quase ótimas e subótimas.
6. Bitstrings e faixas energéticas revelam famílias funcionais.
7. Classes grandes e angularmente dispersas indicam redundância paramétrica.
8. Quase-degenerescência operacional não é degenerescência exata.

In [ ]:
multimodal_indices = mode_table.loc[
    mode_table["estimated_modes"] > 1, "theta_index"
].tolist()

best_bit = (
    valid_bits_df["bitstring"].value_counts().index[0]
    if not valid_bits_df.empty else None
)

print("="*72)
print("RESUMO PARA APRESENTAÇÃO")
print("="*72)
print(f"Execuções: {len(analysis_df):,}")
print(f"Parâmetros: {THETA_WRAPPED.shape[1]}")
print(f"Variância circular média: {CIRCULAR_VARIANCE.mean():.4f}")
print(f"Componentes multimodais: {len(multimodal_indices)} -> {multimodal_indices}")
print(f"PCA: {n90} componentes para 90%")
print(f"PCA: {n95} componentes para 95%")
print(f"KMeans: k={BEST_K}, silhouette={BEST_SILHOUETTE:.4f}")
print(f"Erro < 1e-3: {(energy_error < 1e-3).mean():.2%}")
print(f"Erro < 1e-2: {(energy_error < 1e-2).mean():.2%}")
print(f"Bitstring mais frequente: {best_bit}")
print(f"Fração com bitstring exato: {(analysis_df['bitstring'] == EXACT_BITSTRING).mean():.2%}")

if "equivalence_table" in globals() and not equivalence_table.empty:
    print("\nMaiores classes:")
    print(equivalence_table[
        ["equivalence_class", "size", "mean_angular_spread"]
    ].head(5).to_string(index=False))

print("\nCONCLUSÃO:")
print(
    "O banco contém diversidade paramétrica angular real, mas ela não "
    "necessariamente corresponde a muitas soluções físicas distintas. "
    "A energia e os bitstrings revelam famílias quase ótimas e subótimas, "
    "enquanto classes grandes e dispersas mostram redundância paramétrica."
)

# Conclusão científica

> O banco não deve ser interpretado como 10.000 soluções ótimas independentes. Ele contém uma distribuição estruturada de parâmetros finais, incluindo muitas parametrizações redundantes de poucas famílias funcionais e uma parcela relevante de soluções subótimas. Um Transformer deve ser avaliado pela energia, pelo bitstring e pela distribuição produzida, e não apenas pela distância numérica a um vetor \(\theta\) arbitrário.